# 05 · Activation Functions, Neural Network Units and Expressivity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anuxlab/deep-learning-systems/blob/main/notebooks/05-activations-expressivity/05_activations_expressivity.ipynb)
[![Run on GitHub Actions](https://img.shields.io/badge/run-GitHub%20Actions-2088FF?logo=githubactions&logoColor=white)](https://github.com/anuxlab/deep-learning-systems/actions/workflows/nb-05-activations-expressivity.yml)

| | |
|---|---|
| **Wiki chapter** | [Module 05](https://github.com/anuxlab/deep-learning-systems/wiki/Module-05) |
| **Level** | Beginner–Intermediate |
| **Typical CPU runtime** | ~1 min (GitHub free runner, no GPU) |
| **Prerequisites** | Modules 02, 04 |

## What you will do

- Implement common activations and their derivatives and verify them numerically.
- Understand saturation and why it causes vanishing gradients.
- Implement a stable softmax with temperature.
- See the universal approximation theorem in action, and demonstrate depth-vs-width with an exact $2^d$-piece construction.

> **How to use this notebook.** Read each markdown cell, *predict* what the next code cell will print, then run it.
> Every section ends with `check(...)` assertions: if one fails, something in your understanding (or the code) is off.
> Figures are saved to `figures/` and a machine-readable summary to `metrics.json` next to this notebook.

## 0 · Setup and reproducibility

We import shared helpers (`show` saves each figure to `figures/`, `check` asserts and logs, `timer` logs wall-clock time), fix all random seeds and print an environment report so every run is traceable (see the wiki's *Implementation and Research Standards*).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "_common"))   # shared helpers: notebooks/_common/dlsys.py
from dlsys import *                                       # np, plt, torch, seed_everything, show, check, timer, ...

SEED = seed_everything(42)
ENV = env_report()

## 1 · A zoo of activations
Without a non-linearity, stacked linear layers collapse into one linear map ($W_2W_1=W$). Common choices:

| name | $\phi(z)$ | $\phi'(z)$ |
|---|---|---|
| sigmoid | $1/(1+e^{-z})$ | $\sigma(1-\sigma)$ (max 0.25) |
| tanh | $\tanh z$ | $1-\tanh^2 z$ |
| ReLU | $\max(0,z)$ | $\mathbb 1[z>0]$ |
| Leaky ReLU | $\max(\alpha z,z)$ | $1$ or $\alpha$ |
| softplus | $\log(1+e^z)$ | $\sigma(z)$ |
| GELU | $z\,\Phi(z)$ | $\Phi(z)+z\,\varphi(z)$ |
| SiLU/Swish | $z\,\sigma(z)$ | $\sigma(z)(1+z(1-\sigma(z)))$ |

We implement each with its analytic derivative and compare to a central finite difference.

In [ ]:
from scipy.special import erf, expit
from math import sqrt, pi
act = {
 "sigmoid": (expit, lambda z: expit(z) * (1 - expit(z))),
 "tanh": (np.tanh, lambda z: 1 - np.tanh(z) ** 2),
 "relu": (lambda z: np.maximum(0, z), lambda z: (z > 0).astype(float)),
 "leaky_relu": (lambda z: np.where(z > 0, z, .01 * z), lambda z: np.where(z > 0, 1., .01)),
 "softplus": (lambda z: np.logaddexp(0, z), expit),
 "gelu": (lambda z: z * .5 * (1 + erf(z / sqrt(2))), lambda z: .5 * (1 + erf(z / sqrt(2))) + z * np.exp(-z**2 / 2) / sqrt(2 * pi)),
 "silu": (lambda z: z * expit(z), lambda z: expit(z) * (1 + z * (1 - expit(z)))),
}
z = np.linspace(-5, 5, 1001) + 1e-3     # offset avoids the ReLU kink at exactly 0
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, (f, df) in act.items():
    h = 1e-6; num = (f(z + h) - f(z - h)) / (2 * h)
    check(np.allclose(df(z), num, atol=1e-5), f"{name}: analytic derivative matches finite difference")
    ax[0].plot(z, f(z), label=name); ax[1].plot(z, df(z), label=name)
ax[0].set_ylim(-1.5, 3); ax[0].set_title("φ(z)"); ax[0].legend(ncol=2, fontsize=8); ax[1].set_ylim(-.2, 1.3); ax[1].set_title("φ'(z)")
show("activations")
# cross-check against torch
zt = torch.tensor(z)
check(np.allclose(act["gelu"][0](z), F.gelu(zt).numpy(), atol=1e-8), "our GELU == torch GELU")
check(np.allclose(act["silu"][0](z), F.silu(zt).numpy(), atol=1e-8), "our SiLU == torch SiLU")

## 2 · Saturation and vanishing gradients
Backprop through $L$ layers multiplies $L$ local derivatives. Since $\sigma'\le0.25$, a 20-layer sigmoid chain scales gradients by at most $0.25^{20}\approx10^{-12}$. tanh has $\tanh'(0)=1$ but saturates for $|z|\gg1$; ReLU has gradient exactly 1 on its active side. We measure the **best-case** gradient factor across depth, and a random-input average.

In [ ]:
depths = np.arange(1, 31)
best_sig = 0.25 ** depths
rng = np.random.default_rng(0)
def chain_grad(name, depth, trials=300, scale=1.0):
    f, df = act[name]; out = []
    for _ in range(trials):
        zz = rng.normal(0, scale, size=depth); out.append(np.prod(df(zz)))
    return np.mean(out)
avg = {n: [chain_grad(n, d) for d in depths] for n in ["sigmoid", "tanh", "relu"]}
plt.semilogy(depths, best_sig, "k--", label="sigmoid best-case 0.25^L")
for n, v in avg.items(): plt.semilogy(depths, np.maximum(v, 1e-40), label=f"{n} (avg, z~N(0,1))")
plt.xlabel("depth L"); plt.ylabel("product of local derivatives"); plt.legend(); plt.title("Gradient shrinkage across depth"); show("vanishing_gradient_chain")
check(best_sig[-1] < 1e-17, "sigmoid chain of 30 layers shrinks gradients below 1e-17 even in the best case")
check(avg["relu"][4] > avg["sigmoid"][4] * 100, "ReLU preserves far more gradient than sigmoid at depth 5")

## 3 · Softmax and temperature
$\mathrm{softmax}(z/T)_i=e^{z_i/T}/\sum_je^{z_j/T}$. $T\to0$: one-hot argmax; $T\to\infty$: uniform. Softmax is invariant to adding a constant to all logits — the basis of the stable implementation.

In [ ]:
def softmax(z, T=1.0):
    z = np.asarray(z, float) / T; z = z - z.max(-1, keepdims=True); e = np.exp(z); return e / e.sum(-1, keepdims=True)
z = np.array([2.0, 1.0, 0.1])
for T in [0.1, 0.5, 1, 2, 10]: print(f"T={T:<4}", np.round(softmax(z, T), 4))
check(np.allclose(softmax(z + 123.0), softmax(z)), "shift invariance")
check(np.allclose(softmax(z), F.softmax(torch.tensor(z), -1).numpy()), "matches torch.softmax")
check(softmax(z, 0.01)[0] > .999 and np.ptp(softmax(z, 1e4)) < 1e-3, "T→0 argmax, T→∞ uniform")
# Jacobian of softmax = diag(p) - p p^T (symmetric, rows sum to 0)
p = softmax(z); Jm = np.diag(p) - np.outer(p, p)
check(np.allclose(Jm.sum(1), 0), "softmax Jacobian rows sum to zero (outputs always sum to 1)")

## 4 · Universal approximation in practice
A one-hidden-layer network $\sum_k a_k\,\phi(w_kx+b_k)$ can approximate any continuous function on a compact set to arbitrary accuracy as width grows (Cybenko 1989, Hornik 1991). To see it quickly and deterministically, we **fix random hidden weights** and solve the output layer by least squares (a random-feature model), then vary the width. The target is a wiggly function.

In [ ]:
target = lambda x: np.sin(3 * x) + .5 * np.sin(11 * x) * np.exp(-x**2 / 4)
xs = np.linspace(-3, 3, 600); ys = target(xs)
widths = [2, 4, 8, 16, 32, 64, 128, 256]; errs = {"tanh": [], "relu": []}
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for phi_name, phi in [("tanh", np.tanh), ("relu", lambda z: np.maximum(0, z))]:
    for w_ in widths:
        r = np.random.default_rng(w_)
        Wh, bh = r.normal(0, 3, size=w_), r.uniform(-9, 9, size=w_) if phi_name == "tanh" else r.uniform(-3, 3, size=w_) * r.choice([-3, 3], size=w_)
        Hm = phi(np.outer(xs, Wh) + bh); Hm = np.hstack([Hm, np.ones((len(xs), 1))])
        coef = np.linalg.lstsq(Hm, ys, rcond=None)[0]; fit = Hm @ coef
        errs[phi_name].append(np.sqrt(np.mean((fit - ys) ** 2)))
        if phi_name == "tanh" and w_ in (4, 16, 128): ax[0].plot(xs, fit, label=f"width {w_}")
ax[0].plot(xs, ys, "k--", label="target"); ax[0].legend(); ax[0].set_title("1-hidden-layer tanh fits")
for k, v in errs.items(): ax[1].loglog(widths, v, "o-", label=k)
ax[1].set_xlabel("hidden width"); ax[1].set_ylabel("RMSE"); ax[1].legend(); ax[1].set_title("Approximation error falls with width")
show("universal_approximation")
print({k: [round(e, 4) for e in v] for k, v in errs.items()})
check(errs["tanh"][-1] < errs["tanh"][0] / 20, "tanh error drops >20x from width 2 to 256")
METRICS["uat_rmse_tanh"] = dict(zip(widths, errs["tanh"]))

## 5 · Depth versus width: exponentially many linear pieces
Take the *tent map* $g(x)=2\min(x,1-x)$ on $[0,1]$ — it is computable by a 2-neuron ReLU layer: $g(x)=2\,\mathrm{ReLU}(x)-4\,\mathrm{ReLU}(x-\tfrac12)$. Composing it $d$ times gives a sawtooth with $2^d$ linear pieces using only $2d$ neurons. A *shallow* ReLU net with $N$ hidden units has at most $N+1$ pieces in 1-D, so it needs $\Theta(2^d)$ units to match (Telgarsky 2016). We count the pieces numerically.

In [ ]:
relu = lambda z: np.maximum(0, z)
def tent(x): return 2 * relu(x) - 4 * relu(x - .5)
x = np.linspace(0, 1, 2**12 + 1)
def count_pieces(y):
    slope = np.diff(y) / np.diff(x); change = np.abs(np.diff(slope)) > 1e-6
    return int(change.sum()) + 1
fig, ax = plt.subplots(1, 4, figsize=(14, 2.6)); y = x.copy(); rows = []
for d in range(1, 9):
    y = tent(y); n_pieces = count_pieces(y); rows.append((d, 2 * d, n_pieces))
    print(f"depth {d}: neurons={2*d:3d}  linear pieces={n_pieces:4d}  (2^d = {2**d})")
    check(n_pieces == 2 ** d, f"depth {d} -> exactly 2^{d} pieces")
    if d in (1, 2, 3, 4): ax[d - 1].plot(x, y); ax[d - 1].set_title(f"depth {d}")
show("tent_map_depth")
METRICS["tent_pieces"] = rows

In [ ]:
save_metrics("05", METRICS)
log.info('notebook 05 finished OK')

## Wrap-up

## Exercises (do these next)

1. Plot the 2-D decision regions of a random ReLU MLP and count how many distinct linear regions you find on a grid.
2. Add Mish and ELU to the activation table with derivative checks.
3. Explain why a network with the width-2 tent layer repeated 8 times is hard to *train* even though it can represent a 256-piece function.

## Where to read more

- Wiki chapter: [Module 05](https://github.com/anuxlab/deep-learning-systems/wiki/Module-05) (derivations, worked example, Q&A).
- Cybenko (1989); Hornik (1991); Telgarsky (2016) *Benefits of depth in neural networks*; Hendrycks & Gimpel (2016) GELU.